In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import TimeSeriesSplit, GridSearchCV, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score, 
                             roc_curve, balanced_accuracy_score, precision_score, 
                             recall_score, f1_score, precision_recall_curve, auc)
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold
from sklearn.feature_selection import RFE
from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTEENN
import warnings

warnings.filterwarnings('ignore')

In [ ]:
from IPython.utils import io

with io.capture_output() as captured:
    %run data_preprocessing.ipynb

## Evaluation Strategy: Time Series Cross-Validation

**Why Time Series Cross-Validation Instead of 80/20 Split?**

This notebook uses **TimeSeriesSplit** for more reliable performance estimates:

1. **More Stable Results**: Single train-test split can give misleading results. Cross-validation provides mean ± std across multiple folds.

2. **Better Generalization**: Testing on multiple time periods gives a more reliable estimate of future performance.

3. **Respects Temporal Order**: Each fold trains on past data, tests on future data (no data leakage).

4. **Confidence Intervals**: Standard deviation shows model consistency across time periods.

**How TimeSeriesSplit Works:**
```
Fold 1: Train [====]           Test [=]
Fold 2: Train [========]       Test [=]
Fold 3: Train [============]   Test [=]
Fold 4: Train [================] Test [=]
Fold 5: Train [====================] Test [=]
```

Each fold uses progressively more training data and tests on chronologically later data.

In [ ]:
# Identify coach changes by comparing coaches year-to-year
coaches_by_team_year = (
    coaches_df
    .groupby(['tmID', 'year'])['coachID']
    .agg(lambda x: ','.join(sorted(x.unique())))  # Handle multiple coaches in same year
    .reset_index()
    .sort_values(['tmID', 'year'])
)

# Create next year's coach column
coaches_by_team_year['next_year_coach'] = coaches_by_team_year.groupby('tmID')['coachID'].shift(-1)

# Target: 1 if coach changed, 0 if same coach next year
coaches_by_team_year['coach_changed_next_year'] = (
    coaches_by_team_year['coachID'] != coaches_by_team_year['next_year_coach']
).astype(int)

# Remove rows where we don't have next year data (last year in dataset)
coaches_by_team_year = coaches_by_team_year[coaches_by_team_year['next_year_coach'].notna()].copy()

print(f"Total team-seasons: {len(coaches_by_team_year)}")
print(f"Coach changes: {coaches_by_team_year['coach_changed_next_year'].sum()}")
print(f"No change: {(coaches_by_team_year['coach_changed_next_year'] == 0).sum()}")
print(f"Change rate: {coaches_by_team_year['coach_changed_next_year'].mean():.1%}")

display(coaches_by_team_year.head(10))

### Feature Engineering

The `teams_df` loaded from `data_preprocessing.ipynb` already contains extensive feature engineering:
- Win percentages (overall, home, away, conference)
- Per-game statistics (ppg, papg, pdiffpg, stlpg, blkpg)
- Advanced metrics (ast_to, sbdRpg - defensive metric)
- **3-year and 5-year rolling averages** for all key stats

**What we add here:**
Only features specifically needed for coach change prediction that aren't already in teams_df:
1. **Simple lag features** (previous year) - more interpretable than rolling averages for year-to-year changes
2. **Streak features** - consecutive losing/non-playoff seasons (strong predictors)
3. **Coach features** - tenure with team (low turnover vs high turnover)
4. **Playoff features** - early exits, finals appearances (performance expectations)

In [ ]:
coach_pred_features = teams_df.copy()

if 'made_playoffs' not in coach_pred_features.columns:
    coach_pred_features['made_playoffs'] = (coach_pred_features['playoff'] == 'Y').astype(int)

print(f"Shape: {coach_pred_features.shape}")
display(coach_pred_features[['year', 'tmID', 'name', 'win_pct', 'made_playoffs', 'rank', 'ppg', 'papg']].head())

In [ ]:
# Sort by team and year to ensure proper lagging
coach_pred_features = coach_pred_features.sort_values(['tmID', 'year']).reset_index(drop=True)

# Create only essential lagged features (previous year performance)
# These are more interpretable than rolling averages and capture year-to-year changes
lag_features = ['win_pct', 'made_playoffs', 'rank', 'won', 'ppg', 'papg', 'pdiffpg']

for feature in lag_features:
    coach_pred_features[f'{feature}_prev_year'] = coach_pred_features.groupby('tmID')[feature].shift(1)

# Performance change indicators (most important for coaching decisions)
coach_pred_features['win_pct_change'] = coach_pred_features['win_pct'] - coach_pred_features['win_pct_prev_year']
coach_pred_features['rank_change'] = coach_pred_features['rank'] - coach_pred_features['rank_prev_year']  # Negative = improvement
coach_pred_features['wins_change'] = coach_pred_features['won'] - coach_pred_features['won_prev_year']
coach_pred_features['point_diff_change'] = coach_pred_features['pdiffpg'] - coach_pred_features['pdiffpg_prev_year']

print(" Essential lagged features created (1-year lag)")
print(f"  Features: Previous year performance + year-to-year changes")
print(f"  Note: 3yr and 5yr rolling averages already exist in teams_df")
print(f"\nFeatures with previous year data: {coach_pred_features['win_pct_prev_year'].notna().sum()}")
display(coach_pred_features[['year', 'tmID', 'win_pct', 'win_pct_prev_year', 'win_pct_change', 'rank', 'rank_change']].head(10))

For first-year teams without historical data, we use **simple median imputation** instead of complex year-specific league averages. This is:
- Easier to understand and maintain
- Equally effective in practice (models learn patterns regardless)
- Less prone to overfitting

We create an `is_first_year_team` indicator so models can learn different patterns for expansion teams.

In [ ]:
# For first-year teams, use median imputation
# Create indicator so model knows which teams lack historical data

print("Missing Values in Lagged Features (Before Imputation)")
lag_cols_to_check = [col for col in coach_pred_features.columns if 'prev_year' in col or 'change' in col]
missing_before = coach_pred_features[lag_cols_to_check].isnull().sum()
print(missing_before[missing_before > 0])

# Create indicator: is this team's first year in the league?
coach_pred_features['is_first_year_team'] = (coach_pred_features.groupby('tmID').cumcount() == 0).astype(int)
coach_pred_features['has_prev_year_data'] = coach_pred_features['win_pct_prev_year'].notna().astype(int)

# Simple median imputation for missing lagged values only (not change features)
lag_features_only = [col for col in lag_cols_to_check if 'prev_year' in col]
for col in lag_features_only:
    if coach_pred_features[col].isnull().sum() > 0:
        median_val = coach_pred_features[col].median()
        coach_pred_features[col] = coach_pred_features[col].fillna(median_val)

# Fill change features with 0 for first-year teams
change_features = [col for col in lag_cols_to_check if 'change' in col]
for col in change_features:
    coach_pred_features[col] = coach_pred_features[col].fillna(0)

print("\nMissing Values After Simplified Imputation")
missing_after = coach_pred_features[lag_cols_to_check].isnull().sum()
if missing_after.sum() > 0:
    print(missing_after[missing_after > 0])
else:
    print("All missing values handled with median imputation")

print(f"\n Simplified Imputation Complete")
print(f"  - First-year teams: {coach_pred_features['is_first_year_team'].sum()}")
print(f"  - Used median imputation for lagged features")
print(f"  - Used 0 for change features")
print(f"  - Created 'is_first_year_team' indicator for model awareness")

print("\nSample: First-Year Teams (After Imputation)")
display(coach_pred_features[coach_pred_features['is_first_year_team'] == 1][
    ['year', 'tmID', 'win_pct', 'win_pct_prev_year', 'win_pct_change', 
     'rank', 'rank_change', 'is_first_year_team', 'has_prev_year_data']
].head(10))

Only the most predictive streak patterns for coach changes.

In [ ]:
# Based on EDA, consecutive poor performance is highly correlated with coach changes

# Consecutive losing seasons (win_pct < 0.5)
def count_consecutive_losing(group):
    """Count consecutive seasons with win% < 0.5"""
    streak = 0
    streaks = []
    for val in group:
        if val < 0.5:
            streak += 1
        else:
            streak = 0
        streaks.append(streak)
    return pd.Series(streaks, index=group.index)

coach_pred_features['consecutive_losing_seasons'] = (
    coach_pred_features.groupby('tmID')['win_pct']
    .apply(count_consecutive_losing)
    .reset_index(level=0, drop=True)
)

# Consecutive non-playoff seasons
def count_consecutive_non_playoff(group):
    """Count consecutive seasons without playoffs"""
    streak = 0
    streaks = []
    for val in group:
        if val == 0:
            streak += 1
        else:
            streak = 0
        streaks.append(streak)
    return pd.Series(streaks, index=group.index)

coach_pred_features['consecutive_non_playoff_seasons'] = (
    coach_pred_features.groupby('tmID')['made_playoffs']
    .apply(count_consecutive_non_playoff)
    .reset_index(level=0, drop=True)
)

print(" Simplified streak features created")
print("  - consecutive_losing_seasons (win% < 0.5)")
print("  - consecutive_non_playoff_seasons")
print("  Note: These are the most predictive for coach changes")
display(coach_pred_features[['year', 'tmID', 'win_pct', 'made_playoffs', 
                             'consecutive_losing_seasons', 'consecutive_non_playoff_seasons']].head(15))

In [ ]:
# Handle multiple coaches per team-year by selecting primary coach (most games coached)

coaches_sorted = coaches_df.sort_values(['coachID', 'year'])

# Coach tenure with current team (consecutive years)
coaches_sorted['coach_tenure'] = coaches_sorted.groupby(['coachID', 'tmID']).cumcount() + 1

# For teams with multiple coaches in a year, select the one who coached most games
# Sort by games coached (won + lost) descending, then take first
coaches_sorted['games_coached'] = coaches_sorted['won'] + coaches_sorted['lost']
primary_coaches = (coaches_sorted
                   .sort_values(['year', 'tmID', 'games_coached'], ascending=[True, True, False])
                   .groupby(['year', 'tmID'])
                   .first()
                   .reset_index())

# Merge only essential coach features from primary coach
coach_pred_features = coach_pred_features.merge(
    primary_coaches[['year', 'tmID', 'coachID', 'coach_tenure']],
    on=['year', 'tmID'],
    how='left'
)

# Coach first year with team indicator
coach_pred_features['coach_is_new'] = (coach_pred_features['coach_tenure'] == 1).astype(int)

# Fill missing coach data with defaults
coach_pred_features['coach_tenure'] = coach_pred_features['coach_tenure'].fillna(1).astype(int)
coach_pred_features['coach_is_new'] = coach_pred_features['coach_is_new'].fillna(1).astype(int)

print(" Simplified coach features added")
print(f"  - coach_tenure: Years with current team")
print(f"  - coach_is_new: First year indicator")
print(f"  - Handles multiple coaches by selecting primary (most games)")
print(f"\nTeams with coach data: {coach_pred_features['coachID'].notna().sum()}")
display(coach_pred_features[['year', 'tmID', 'coachID', 'coach_tenure', 'coach_is_new']].head(10))

In [ ]:
# Championship/Finals appearance (strong positive indicator)
coach_pred_features['reached_finals'] = coach_pred_features['finals'].notna().astype(int)
coach_pred_features['won_championship'] = (coach_pred_features['finals'] == 'W').astype(int)

print(" Simplified playoff features added")
print("  - reached_finals: Team made it to finals")
print("  - won_championship: Won championship")
print("  Note: These are the most predictive playoff-related features")
display(coach_pred_features[['year', 'tmID', 'made_playoffs', 
                             'reached_finals', 'won_championship']].head(10))

In [ ]:
# Merge features with target variable
coach_change_dataset = coach_pred_features.merge(
    coaches_by_team_year[['year', 'tmID', 'coach_changed_next_year']],
    on=['year', 'tmID'],
    how='inner'  # Only keep rows where we have the target
)

print(f"Final dataset shape: {coach_change_dataset.shape}")
print(f"Total samples: {len(coach_change_dataset)}")
print(f"Coach changes: {coach_change_dataset['coach_changed_next_year'].sum()}")
print(f"No changes: {(coach_change_dataset['coach_changed_next_year'] == 0).sum()}")
print(f"Class balance: {coach_change_dataset['coach_changed_next_year'].mean():.1%} changed")

# Check for missing values in key features
print("\nMissing Values in Key Features")
key_features = [
    'win_pct', 'rank', 'made_playoffs', 'won', 'lost',
    'win_pct_prev_year', 'rank_prev_year', 'made_playoffs_prev_year',
    'win_pct_change', 'rank_change', 
    'consecutive_losing_seasons', 'consecutive_non_playoff_seasons',
    'coach_tenure'
]
missing_summary = coach_change_dataset[key_features].isnull().sum()
if missing_summary.sum() > 0:
    display(missing_summary[missing_summary > 0])
else:
    print(" No missing values in key features")

print(f"\nSample of Final Dataset")
display(coach_change_dataset[['year', 'tmID', 'name', 'win_pct', 'rank', 'made_playoffs', 
                               'win_pct_prev_year', 'rank_change', 'consecutive_losing_seasons',
                               'coach_tenure', 'coach_changed_next_year']].head(15))

## Feature Selection

In [ ]:
# Select features for modeling (exclude identifiers, target, and redundant columns)
exclude_from_model = [
    'year', 'tmID', 'name', 'lgID', 'coachID',  # Identifiers
    'coach_changed_next_year',  # Target
    'playoff', 'confID', 'divID', 'seeded', 'firstRound', 'semis', 'finals',  # Categorical/redundant
    'GP', 'homeW', 'homeL', 'awayW', 'awayL', 'confW', 'confL',  # Raw counts (we have rates)
    'min', 'attend',  # Not predictive
    'o_fgm', 'd_fgm', 'o_fga', 'd_fga', 'o_ftm', 'd_ftm', 'o_fta', 'd_fta',  # Raw shooting stats
    'o_3pm', 'd_3pm', 'o_3pa', 'd_3pa',  # Raw 3-point stats
    'o_oreb', 'd_oreb', 'o_dreb', 'd_dreb',  # Raw rebound stats (we have o_reb, d_reb)
    'o_asts', 'd_asts', 'o_pf', 'd_pf', 'o_stl', 'd_stl', 'o_to', 'd_to', 'o_blk', 'd_blk'  # Raw stats (we have per-game)
]

# Get all numeric columns
all_numeric_cols = coach_change_dataset.select_dtypes(include=[np.number]).columns.tolist()

# Filter out excluded columns
model_features = [col for col in all_numeric_cols if col not in exclude_from_model]
print(f"Number of model_features: {len(model_features)}")

In [ ]:
# Calculate correlation of all features with target
feature_correlations = coach_change_dataset[model_features + ['coach_changed_next_year']].corr()['coach_changed_next_year'].drop('coach_changed_next_year')

# Sort by absolute correlation
feature_correlations_abs = feature_correlations.abs().sort_values(ascending=False)

# Select top features based on:
# 1. High Variance
# 2. High correlation with target
# 3. Low multicollinearity with each other
# 4. Recursive Feature Elimination

# Remove low-variance features first
selector = VarianceThreshold(threshold=0.01)
X_var = selector.fit_transform(coach_change_dataset[model_features])
retained_features = [f for f, keep in zip(model_features, selector.get_support()) if keep]

print(f"Number of features after VarianceThreshold: {len(retained_features)}")

# Now compute correlations only on retained features
feature_correlations = coach_change_dataset[retained_features + ['coach_changed_next_year']].corr()['coach_changed_next_year'].drop('coach_changed_next_year')
feature_correlations_abs = feature_correlations.abs().sort_values(ascending=False)

# Select top 40 features by absolute correlation
top_corr_features = feature_correlations_abs[feature_correlations_abs.notna()].head(30).index.tolist()
selected_features = top_corr_features

# Filter to only features that exist in model_features
selected_features = [f for f in selected_features if f in model_features]

print(f"\nSELECTED FEATURES FOR SIMPLIFIED MODEL")
print(f"Total selected: {len(selected_features)}")
print("\nFeatures:")
for i, feat in enumerate(selected_features, 1):
    corr_val = feature_correlations[feat] if feat in feature_correlations.index else 0
    print(f"{i:2d}. {feat:40s} {corr_val:+.4f}")

# Check multicollinearity
print("\nCHECKING MULTICOLLINEARITY")
selected_corr_matrix = coach_change_dataset[selected_features].corr()
high_corr_pairs = []
for i in range(len(selected_features)):
    for j in range(i+1, len(selected_features)):
        corr_val = abs(selected_corr_matrix.iloc[i, j])
        if corr_val > 0.8:  # Threshold for high correlation
            high_corr_pairs.append((selected_features[i], selected_features[j], corr_val))

if high_corr_pairs:
    print("  High correlation pairs (>0.8):")
    for feat1, feat2, corr_val in high_corr_pairs:
        print(f"  {feat1} <-> {feat2}: {corr_val:.3f}")
else:
    print("No highly correlated feature pairs (all < 0.8)")

# Automatically drop the second feature in each highly correlated pair
features_to_drop = set([pair[1] for pair in high_corr_pairs])
selected_features = [f for f in selected_features if f not in features_to_drop]

# Use RandomForest for RFE
rf_for_rfe = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
rfe = RFE(rf_for_rfe, n_features_to_select=7)
X_rfe = coach_change_dataset[selected_features]
y_rfe = coach_change_dataset['coach_changed_next_year']

rfe.fit(X_rfe, y_rfe)
final_features = [feat for feat, keep in zip(selected_features, rfe.support_) if keep]

print("\nFINAL FEATURES AFTER RFE (Random Forest, n=5):")
for i, feat in enumerate(final_features, 1):
    print(f"{i:2d}. {feat}")

# Use only these features for the rest of your modeling
selected_features = final_features

## Prediction

In [ ]:
# Use TimeSeriesSplit with 5 folds
n_splits = 5

In [ ]:
# Prepare features and target
X_engineered = coach_change_dataset[model_features].copy()
y_engineered = coach_change_dataset['coach_changed_next_year'].copy()

# Handle any remaining missing values (should be minimal after our preprocessing)
missing_counts = X_engineered.isnull().sum()
if missing_counts.sum() > 0:
    print("=== Handling remaining missing values ===")
    print(missing_counts[missing_counts > 0])
    X_engineered = X_engineered.fillna(X_engineered.median())
    print("Filled with median values")
else:
    print("No missing values in feature set")

# Sort data by year for time series split
X_engineered_sorted = X_engineered.join(coach_change_dataset[['year']]).sort_values('year')
y_engineered_sorted = y_engineered[X_engineered_sorted.index]
X_engineered_sorted = X_engineered_sorted.drop('year', axis=1)

In [ ]:
# Prepare data with selected features only (sorted by year for time series CV)
X_selected_sorted = X_engineered_sorted[selected_features]

print(f"=== TRAINING WITH SELECTED FEATURES (CROSS-VALIDATION) ===")
print(f"Total samples: {len(X_selected_sorted)}")
print(f"Number of features: {len(selected_features)}")
print(f"Feature-to-sample ratio: 1:{len(X_selected_sorted) // len(selected_features)}")

print("\n" + "="*70)
print("LOGISTIC REGRESSION - SELECTED FEATURES (CROSS-VALIDATION)")
print("="*70)

lr_selected = LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000, C=1.0)

# Use same time series split as other approaches
tscv_sel = TimeSeriesSplit(n_splits=n_splits)

# Store results for each fold
lr_cv_results_sel = {'roc_auc': [], 'balanced_accuracy': [], 'precision': [], 'recall': [], 'f1': []}
lr_confusion_matrices_sel = []
lr_feature_importances_sel = []
last_fold_lr_data_sel = {}  # For ROC curve plotting

for fold, (train_idx, test_idx) in enumerate(tscv_sel.split(X_selected_sorted), 1):
    # Split data
    X_train_fold = X_selected_sorted.iloc[train_idx]
    X_test_fold = X_selected_sorted.iloc[test_idx]
    y_train_fold = y_engineered_sorted.iloc[train_idx]
    y_test_fold = y_engineered_sorted.iloc[test_idx]
    
    # Scale features (fit on train, transform both)
    scaler_sel = StandardScaler()
    X_train_scaled = scaler_sel.fit_transform(X_train_fold)
    X_test_scaled = scaler_sel.transform(X_test_fold)
    
    # Train model
    lr_selected.fit(X_train_scaled, y_train_fold)
    
    # Predictions
    y_pred = lr_selected.predict(X_test_scaled)
    y_pred_proba = lr_selected.predict_proba(X_test_scaled)[:, 1]
    
    # Calculate metrics
    lr_cv_results_sel['roc_auc'].append(roc_auc_score(y_test_fold, y_pred_proba))
    lr_cv_results_sel['balanced_accuracy'].append(balanced_accuracy_score(y_test_fold, y_pred))
    lr_cv_results_sel['precision'].append(precision_score(y_test_fold, y_pred, zero_division=0))
    lr_cv_results_sel['recall'].append(recall_score(y_test_fold, y_pred, zero_division=0))
    lr_cv_results_sel['f1'].append(f1_score(y_test_fold, y_pred, zero_division=0))
    
    # Store confusion matrix and feature importance
    lr_confusion_matrices_sel.append(confusion_matrix(y_test_fold, y_pred))
    lr_feature_importances_sel.append(lr_selected.coef_[0])
    
    # Save last fold's data for ROC curve
    if fold == n_splits:
        last_fold_lr_data_sel = {'y_test': y_test_fold, 'y_pred_proba': y_pred_proba}
    
    print(f"Fold {fold}: ROC-AUC={lr_cv_results_sel['roc_auc'][-1]:.4f}, "
          f"Balanced Acc={lr_cv_results_sel['balanced_accuracy'][-1]:.4f}, "
          f"F1={lr_cv_results_sel['f1'][-1]:.4f}")

# Calculate mean and std
print("\n--- Cross-Validation Results (Logistic Regression - Selected) ---")
for metric_name, scores in lr_cv_results_sel.items():
    print(f"{metric_name.upper():20s}: {np.mean(scores):.4f} (+/- {np.std(scores):.4f})")

# Average confusion matrix
avg_cm_lr_sel = np.mean(lr_confusion_matrices_sel, axis=0).astype(int)
print(f"\n--- Average Confusion Matrix ---")
print(avg_cm_lr_sel)
print(f"True Negatives: {avg_cm_lr_sel[0,0]}, False Positives: {avg_cm_lr_sel[0,1]}")
print(f"False Negatives: {avg_cm_lr_sel[1,0]}, True Positives: {avg_cm_lr_sel[1,1]}")

# Average feature importance
avg_feature_importance_lr_sel = np.mean(lr_feature_importances_sel, axis=0)
feature_importance_lr_sel = pd.DataFrame({
    'feature': selected_features,
    'coefficient': avg_feature_importance_lr_sel,
    'abs_coefficient': np.abs(avg_feature_importance_lr_sel)
}).sort_values('abs_coefficient', ascending=False)

print("\n--- Feature Importance (Average across folds) ---")
display(feature_importance_lr_sel)

# Create backward-compatible variables
roc_auc_lr_sel = np.mean(lr_cv_results_sel['roc_auc'])
cm_lr_sel = avg_cm_lr_sel

print("\n" + "="*70)
print("RANDOM FOREST - SELECTED FEATURES (CROSS-VALIDATION)")
print("="*70)

rf_selected = RandomForestClassifier(
    n_estimators=100, class_weight='balanced', random_state=42,
    max_depth=5, min_samples_split=15, min_samples_leaf=8, max_features='sqrt'
)

# Store results for each fold
rf_cv_results_sel = {'roc_auc': [], 'balanced_accuracy': [], 'precision': [], 'recall': [], 'f1': []}
rf_confusion_matrices_sel = []
rf_feature_importances_sel = []
last_fold_rf_data_sel = {}  # For ROC curve plotting

for fold, (train_idx, test_idx) in enumerate(tscv_sel.split(X_selected_sorted), 1):
    # Split data
    X_train_fold = X_selected_sorted.iloc[train_idx]
    X_test_fold = X_selected_sorted.iloc[test_idx]
    y_train_fold = y_engineered_sorted.iloc[train_idx]
    y_test_fold = y_engineered_sorted.iloc[test_idx]
    
    # Train model (no scaling needed for Random Forest)
    rf_selected.fit(X_train_fold, y_train_fold)
    
    # Predictions
    y_pred = rf_selected.predict(X_test_fold)
    y_pred_proba = rf_selected.predict_proba(X_test_fold)[:, 1]
    
    # Calculate metrics
    rf_cv_results_sel['roc_auc'].append(roc_auc_score(y_test_fold, y_pred_proba))
    rf_cv_results_sel['balanced_accuracy'].append(balanced_accuracy_score(y_test_fold, y_pred))
    rf_cv_results_sel['precision'].append(precision_score(y_test_fold, y_pred, zero_division=0))
    rf_cv_results_sel['recall'].append(recall_score(y_test_fold, y_pred, zero_division=0))
    rf_cv_results_sel['f1'].append(f1_score(y_test_fold, y_pred, zero_division=0))
    
    # Store confusion matrix and feature importance
    rf_confusion_matrices_sel.append(confusion_matrix(y_test_fold, y_pred))
    rf_feature_importances_sel.append(rf_selected.feature_importances_)
    
    # Save last fold's data for ROC curve
    if fold == n_splits:
        last_fold_rf_data_sel = {'y_test': y_test_fold, 'y_pred_proba': y_pred_proba}
    
    print(f"Fold {fold}: ROC-AUC={rf_cv_results_sel['roc_auc'][-1]:.4f}, "
          f"Balanced Acc={rf_cv_results_sel['balanced_accuracy'][-1]:.4f}, "
          f"F1={rf_cv_results_sel['f1'][-1]:.4f}")

# Calculate mean and std
print("\n--- Cross-Validation Results (Random Forest - Selected) ---")
for metric_name, scores in rf_cv_results_sel.items():
    print(f"{metric_name.upper():20s}: {np.mean(scores):.4f} (+/- {np.std(scores):.4f})")

# Average confusion matrix
avg_cm_rf_sel = np.mean(rf_confusion_matrices_sel, axis=0).astype(int)
print(f"\n--- Average Confusion Matrix ---")
print(avg_cm_rf_sel)
print(f"True Negatives: {avg_cm_rf_sel[0,0]}, False Positives: {avg_cm_rf_sel[0,1]}")
print(f"False Negatives: {avg_cm_rf_sel[1,0]}, True Positives: {avg_cm_rf_sel[1,1]}")

# Average feature importance
avg_feature_importance_rf_sel = np.mean(rf_feature_importances_sel, axis=0)
feature_importance_rf_sel = pd.DataFrame({
    'feature': selected_features,
    'importance': avg_feature_importance_rf_sel
}).sort_values('importance', ascending=False)

print("\n--- Feature Importance (Average across folds) ---")
display(feature_importance_rf_sel)

# Create backward-compatible variables
roc_auc_rf_sel = np.mean(rf_cv_results_sel['roc_auc'])
cm_rf_sel = avg_cm_rf_sel

print("\n" + "="*70)
print("GRADIENT BOOSTING - SELECTED FEATURES (CROSS-VALIDATION)")
print("="*70)

gb_selected = GradientBoostingClassifier(
    n_estimators=100, learning_rate=0.1, max_depth=3, 
    min_samples_split=15, min_samples_leaf=8, subsample=0.8,
    random_state=42
)

# Store results for each fold
gb_cv_results_sel = {'roc_auc': [], 'balanced_accuracy': [], 'precision': [], 'recall': [], 'f1': []}
gb_confusion_matrices_sel = []
gb_feature_importances_sel = []
last_fold_gb_data_sel = {}  # For ROC curve plotting

for fold, (train_idx, test_idx) in enumerate(tscv_sel.split(X_selected_sorted), 1):
    # Split data
    X_train_fold = X_selected_sorted.iloc[train_idx]
    X_test_fold = X_selected_sorted.iloc[test_idx]
    y_train_fold = y_engineered_sorted.iloc[train_idx]
    y_test_fold = y_engineered_sorted.iloc[test_idx]
    
    # Train model (no scaling needed for Gradient Boosting)
    gb_selected.fit(X_train_fold, y_train_fold)
    
    # Predictions
    y_pred = gb_selected.predict(X_test_fold)
    y_pred_proba = gb_selected.predict_proba(X_test_fold)[:, 1]
    
    # Calculate metrics
    gb_cv_results_sel['roc_auc'].append(roc_auc_score(y_test_fold, y_pred_proba))
    gb_cv_results_sel['balanced_accuracy'].append(balanced_accuracy_score(y_test_fold, y_pred))
    gb_cv_results_sel['precision'].append(precision_score(y_test_fold, y_pred, zero_division=0))
    gb_cv_results_sel['recall'].append(recall_score(y_test_fold, y_pred, zero_division=0))
    gb_cv_results_sel['f1'].append(f1_score(y_test_fold, y_pred, zero_division=0))
    
    # Store confusion matrix and feature importance
    gb_confusion_matrices_sel.append(confusion_matrix(y_test_fold, y_pred))
    gb_feature_importances_sel.append(gb_selected.feature_importances_)
    
    # Save last fold's data for ROC curve
    if fold == n_splits:
        last_fold_gb_data_sel = {'y_test': y_test_fold, 'y_pred_proba': y_pred_proba}
    
    print(f"Fold {fold}: ROC-AUC={gb_cv_results_sel['roc_auc'][-1]:.4f}, "
          f"Balanced Acc={gb_cv_results_sel['balanced_accuracy'][-1]:.4f}, "
          f"F1={gb_cv_results_sel['f1'][-1]:.4f}")

# Calculate mean and std
print("\n--- Cross-Validation Results (Gradient Boosting - Selected) ---")
for metric_name, scores in gb_cv_results_sel.items():
    print(f"{metric_name.upper():20s}: {np.mean(scores):.4f} (+/- {np.std(scores):.4f})")

# Average confusion matrix
avg_cm_gb_sel = np.mean(gb_confusion_matrices_sel, axis=0).astype(int)
print(f"\n--- Average Confusion Matrix ---")
print(avg_cm_gb_sel)
print(f"True Negatives: {avg_cm_gb_sel[0,0]}, False Positives: {avg_cm_gb_sel[0,1]}")
print(f"False Negatives: {avg_cm_gb_sel[1,0]}, True Positives: {avg_cm_gb_sel[1,1]}")

# Average feature importance
avg_feature_importance_gb_sel = np.mean(gb_feature_importances_sel, axis=0)
feature_importance_gb_sel = pd.DataFrame({
    'feature': selected_features,
    'importance': avg_feature_importance_gb_sel
}).sort_values('importance', ascending=False)

print("\n--- Feature Importance (Average across folds) ---")
display(feature_importance_gb_sel)

# Create backward-compatible variables
roc_auc_gb_sel = np.mean(gb_cv_results_sel['roc_auc'])
cm_gb_sel = avg_cm_gb_sel

print("\n" + "="*70)
print("SUPPORT VECTOR MACHINE - SELECTED FEATURES (CROSS-VALIDATION)")
print("="*70)

svm_selected = SVC(
    kernel='rbf', C=1.0, gamma='scale', 
    class_weight='balanced', probability=True, random_state=42
)

# Store results for each fold
svm_cv_results_sel = {'roc_auc': [], 'balanced_accuracy': [], 'precision': [], 'recall': [], 'f1': []}
svm_confusion_matrices_sel = []
last_fold_svm_data_sel = {}  # For ROC curve plotting

for fold, (train_idx, test_idx) in enumerate(tscv_sel.split(X_selected_sorted), 1):
    # Split data
    X_train_fold = X_selected_sorted.iloc[train_idx]
    X_test_fold = X_selected_sorted.iloc[test_idx]
    y_train_fold = y_engineered_sorted.iloc[train_idx]
    y_test_fold = y_engineered_sorted.iloc[test_idx]
    
    # Scale features (required for SVM)
    scaler_svm = StandardScaler()
    X_train_scaled = scaler_svm.fit_transform(X_train_fold)
    X_test_scaled = scaler_svm.transform(X_test_fold)
    
    # Train model
    svm_selected.fit(X_train_scaled, y_train_fold)
    
    # Predictions
    y_pred = svm_selected.predict(X_test_scaled)
    y_pred_proba = svm_selected.predict_proba(X_test_scaled)[:, 1]
    
    # Calculate metrics
    svm_cv_results_sel['roc_auc'].append(roc_auc_score(y_test_fold, y_pred_proba))
    svm_cv_results_sel['balanced_accuracy'].append(balanced_accuracy_score(y_test_fold, y_pred))
    svm_cv_results_sel['precision'].append(precision_score(y_test_fold, y_pred, zero_division=0))
    svm_cv_results_sel['recall'].append(recall_score(y_test_fold, y_pred, zero_division=0))
    svm_cv_results_sel['f1'].append(f1_score(y_test_fold, y_pred, zero_division=0))
    
    # Store confusion matrix
    svm_confusion_matrices_sel.append(confusion_matrix(y_test_fold, y_pred))
    
    # Save last fold's data for ROC curve
    if fold == n_splits:
        last_fold_svm_data_sel = {'y_test': y_test_fold, 'y_pred_proba': y_pred_proba}
    
    print(f"Fold {fold}: ROC-AUC={svm_cv_results_sel['roc_auc'][-1]:.4f}, "
          f"Balanced Acc={svm_cv_results_sel['balanced_accuracy'][-1]:.4f}, "
          f"F1={svm_cv_results_sel['f1'][-1]:.4f}")

# Calculate mean and std
print("\n--- Cross-Validation Results (SVM - Selected) ---")
for metric_name, scores in svm_cv_results_sel.items():
    print(f"{metric_name.upper():20s}: {np.mean(scores):.4f} (+/- {np.std(scores):.4f})")

# Average confusion matrix
avg_cm_svm_sel = np.mean(svm_confusion_matrices_sel, axis=0).astype(int)
print(f"\n--- Average Confusion Matrix ---")
print(avg_cm_svm_sel)
print(f"True Negatives: {avg_cm_svm_sel[0,0]}, False Positives: {avg_cm_svm_sel[0,1]}")
print(f"False Negatives: {avg_cm_svm_sel[1,0]}, True Positives: {avg_cm_svm_sel[1,1]}")

print("\nNote: SVM does not provide feature importances directly.")
print("Use permutation importance or SHAP values for SVM feature importance analysis.")

# Create backward-compatible variables
roc_auc_svm_sel = np.mean(svm_cv_results_sel['roc_auc'])
cm_svm_sel = avg_cm_svm_sel


In [ ]:
print("\n" + "="*70)
print("MODEL COMPARISON SUMMARY")
print("="*70)

# Compile results from all models
model_results = {
    'Logistic Regression': lr_cv_results_sel,
    'Random Forest': rf_cv_results_sel,
    'Gradient Boosting': gb_cv_results_sel,
    'SVM (RBF)': svm_cv_results_sel
}

print("\nPerformance Comparison (Mean ± Std):")
print("-" * 100)
print(f"{'Model':<22} {'ROC-AUC':>16} {'Balanced Acc':>16} {'Precision':>16} {'Recall':>16} {'F1-Score':>16}")
print("-" * 100)

for model_name, results in model_results.items():
    print(f"{model_name:<22} ", end="")
    for metric in ['roc_auc', 'balanced_accuracy', 'precision', 'recall', 'f1']:
        mean_val = np.mean(results[metric])
        std_val = np.std(results[metric])
        print(f"{mean_val:.3f}±{std_val:.3f}    ", end="")
    print()

print("-" * 100)

# Find best model for each metric
print("\n\nBest Model by Metric:")
for metric in ['roc_auc', 'balanced_accuracy', 'precision', 'recall', 'f1']:
    best_model = max(model_results.items(), key=lambda x: np.mean(x[1][metric]))
    best_score = np.mean(best_model[1][metric])
    print(f"  {metric.upper():20s}: {best_model[0]:25s} ({best_score:.4f})")

# print("\n" + "="*70)
# print("KEY INSIGHTS")
# print("="*70)

# print("\n1. CHALLENGES:")
# print("   - Small dataset (~140 total samples)")
# print("   - Coach changes depend on non-statistical factors:")
# print("     * Ownership decisions, contract situations, media/fan pressure, player relationships")
# print("   - Performance stats alone can only explain PART of the story")

# print("\n2. FEATURE ENGINEERING LESSONS:")
# print("   - Too many features for small dataset caused OVERFITTING")
# print("   - Feature-to-sample ratio of 1:2 is far too high (should be 1:10+)")
# print("   - Feature selection improved performance by reducing overfitting")
# print("   - Simpler models often perform better with limited data")

# print("\n3. MOST IMPORTANT FEATURES (FROM LOGISTIC REGRESSION):")
# if len(feature_importance_lr_sel) > 0:
#     for idx, row in feature_importance_lr_sel.head(5).iterrows():
#         direction = "↑" if row['coefficient'] > 0 else "↓"
#         print(f"   {direction} {row['feature']:30s} (coef: {row['coefficient']:+.3f})")

# print("\n4. MODEL COMPARISON:")
# print("   - Logistic Regression: Interpretable, works well with limited data")
# print("   - Random Forest: Handles non-linear relationships, provides feature importance")
# print("   - Gradient Boosting: Often best performance, sequential error correction")
# print("   - SVM: Effective in high-dimensional space, robust to outliers")

# print("\n5. CROSS-VALIDATION BENEFITS:")
# print(f"   - Used {n_splits}-fold time-series cross-validation")
# print("   - More stable and reliable performance estimates")
# print("   - Confidence intervals show model consistency across time periods")
# print("   - Better assessment of generalization to future years")


## Addressing Class Imbalance with SMOTE

**Problem**: ~40% coach changes vs 60% no changes creates bias toward majority class.

**Possible Solution**: SMOTE (Synthetic Minority Over-sampling Technique) generates synthetic minority samples.

**Key Implementation Details**:
- Apply SMOTE **inside each CV fold** (prevents data leakage)
- Use conservative k_neighbors=3 for small dataset
- Add threshold optimization to find best precision-recall trade-off
- Compare baseline vs SMOTE results

In [ ]:
# Train models with SMOTE - efficient loop structure
print("\n" + "="*70)
print("MODELS WITH SMOTE (CLASS IMBALANCE CORRECTION)")
print("="*70)

models_smote = {
    'Logistic Regression': LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000, C=1.0),
    'Random Forest': RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42,
                                           max_depth=5, min_samples_split=15, min_samples_leaf=8, max_features='sqrt'),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3,
                                                     min_samples_split=15, min_samples_leaf=8, subsample=0.8, random_state=42),
    'SVM (RBF)': SVC(kernel='rbf', C=1.0, gamma='scale', class_weight='balanced', probability=True, random_state=42)
}

# Models requiring scaling
needs_scaling = {'Logistic Regression', 'SVM (RBF)'}

# Store results
smote_results = {}
smote_pr_data = {}  # For precision-recall curves

for model_name, model in models_smote.items():
    print(f"\n{model_name}:")
    cv_results = {'roc_auc': [], 'balanced_accuracy': [], 'precision': [], 'recall': [], 'f1': [], 'pr_auc': []}
    confusion_matrices = []
    pr_curves = []
    
    for fold, (train_idx, test_idx) in enumerate(tscv_sel.split(X_selected_sorted), 1):
        X_train_fold = X_selected_sorted.iloc[train_idx]
        X_test_fold = X_selected_sorted.iloc[test_idx]
        y_train_fold = y_engineered_sorted.iloc[train_idx]
        y_test_fold = y_engineered_sorted.iloc[test_idx]
        
        # Apply SMOTE to training data only (prevent leakage)
        smote = SMOTE(k_neighbors=3, random_state=42)
        X_train_smote, y_train_smote = smote.fit_resample(X_train_fold, y_train_fold)
        
        # Scale if needed
        if model_name in needs_scaling:
            scaler = StandardScaler()
            X_train_smote = scaler.fit_transform(X_train_smote)
            X_test_scaled = scaler.transform(X_test_fold)
        else:
            X_test_scaled = X_test_fold
        
        # Train and predict
        model.fit(X_train_smote, y_train_smote)
        y_pred = model.predict(X_test_scaled)
        y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]
        
        # Metrics
        cv_results['roc_auc'].append(roc_auc_score(y_test_fold, y_pred_proba))
        cv_results['balanced_accuracy'].append(balanced_accuracy_score(y_test_fold, y_pred))
        cv_results['precision'].append(precision_score(y_test_fold, y_pred, zero_division=0))
        cv_results['recall'].append(recall_score(y_test_fold, y_pred, zero_division=0))
        cv_results['f1'].append(f1_score(y_test_fold, y_pred, zero_division=0))
        
        # Precision-Recall AUC
        precision_curve, recall_curve, _ = precision_recall_curve(y_test_fold, y_pred_proba)
        pr_auc = auc(recall_curve, precision_curve)
        cv_results['pr_auc'].append(pr_auc)
        
        confusion_matrices.append(confusion_matrix(y_test_fold, y_pred))
        
        # Store last fold for PR curve
        if fold == n_splits:
            pr_curves.append((precision_curve, recall_curve, y_test_fold, y_pred_proba))
        
        print(f"  Fold {fold}: ROC-AUC={cv_results['roc_auc'][-1]:.4f}, "
              f"PR-AUC={cv_results['pr_auc'][-1]:.4f}, F1={cv_results['f1'][-1]:.4f}")
    
    # Store results
    smote_results[model_name] = cv_results
    smote_pr_data[model_name] = pr_curves[0] if pr_curves else None
    
    # Summary
    print(f"  Mean ROC-AUC: {np.mean(cv_results['roc_auc']):.4f} (±{np.std(cv_results['roc_auc']):.4f})")
    print(f"  Mean PR-AUC:  {np.mean(cv_results['pr_auc']):.4f} (±{np.std(cv_results['pr_auc']):.4f})")
    print(f"  Mean Recall:  {np.mean(cv_results['recall']):.4f} (±{np.std(cv_results['recall']):.4f})")
    print(f"  Mean F1:      {np.mean(cv_results['f1']):.4f} (±{np.std(cv_results['f1']):.4f})")
    
    avg_cm = np.mean(confusion_matrices, axis=0).astype(int)
    print(f"  Avg Confusion: TN={avg_cm[0,0]}, FP={avg_cm[0,1]}, FN={avg_cm[1,0]}, TP={avg_cm[1,1]}")

print("\n" + "="*70)

In [ ]:
# Precision-Recall Curves
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()

for idx, (model_name, pr_data) in enumerate(smote_pr_data.items()):
    if pr_data:
        precision_curve, recall_curve, y_test, y_pred_proba = pr_data
        
        # Plot PR curve
        axes[idx].plot(recall_curve, precision_curve, linewidth=2, label=f'PR curve (AUC = {auc(recall_curve, precision_curve):.3f})')
        axes[idx].plot([0, 1], [y_test.mean(), y_test.mean()], 'k--', label=f'Baseline ({y_test.mean():.3f})')
        axes[idx].set_xlabel('Recall', fontsize=11)
        axes[idx].set_ylabel('Precision', fontsize=11)
        axes[idx].set_title(f'{model_name}', fontsize=12, fontweight='bold')
        axes[idx].legend(loc='best')
        axes[idx].grid(True, alpha=0.3)
        axes[idx].set_xlim([0, 1])
        axes[idx].set_ylim([0, 1])

plt.suptitle('Precision-Recall Curves (SMOTE Models - Last Fold)', fontsize=14, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

print("Note: PR curves are more informative than ROC curves for imbalanced datasets.")

In [ ]:
# Baseline vs SMOTE Comparison
print("\n" + "="*70)
print("BASELINE vs SMOTE COMPARISON")
print("="*70)

comparison_metrics = ['roc_auc', 'pr_auc', 'balanced_accuracy', 'precision', 'recall', 'f1']

for model_name in model_results.keys():
    print(f"\n{model_name}:")
    print(f"  {'Metric':<20} {'Baseline':>12} {'SMOTE':>12} {'Improvement':>12}")
    print("  " + "-" * 58)
    
    for metric in comparison_metrics:
        if metric == 'pr_auc':
            baseline_val = 0  # Not computed for baseline
            baseline_str = "N/A"
        else:
            baseline_val = np.mean(model_results[model_name][metric])
            baseline_str = f"{baseline_val:.4f}"
        
        smote_val = np.mean(smote_results[model_name][metric])
        
        if baseline_val > 0:
            improvement = ((smote_val - baseline_val) / baseline_val) * 100
            improvement_str = f"{improvement:+.1f}%"
        else:
            improvement_str = "N/A"
        
        print(f"  {metric.upper():<20} {baseline_str:>12} {smote_val:>12.4f} {improvement_str:>12}")

print("\n" + "="*70)
print("KEY FINDINGS:")
print("  SMOTE improves recall (catches more coach changes)")
print("  PR-AUC provides better evaluation for imbalanced data")
print("  Trade-off: Slightly lower precision, significantly higher recall")
print("  F1-Score balances both metrics")
print("="*70)

### Why applying SMOTE actually made some results worse?

- 40% vs 60% split is actually fairly balanced (not extreme like 5% vs 95%). class_weight='balanced' was already handling it effectively and adding SMOTE was overkill that introduced noise
- Only ~140 samples is not enough data for SMOTE to learn real patterns. Synthetic samples created artificial patterns that don't exist in reality -> Models learned these fake patterns -> worse generalization
- Coach changes have high variance within the minority class: some fired after terrible seasons, some fired after good seasons (ownership disputes, contracts), some kept despite poor performance (politics, patience) -> SMOTE can't replicate this complex, non-uniform behavior

We'll archive this findings and move on.

## Hyperparameter Tuning

Now let's optimize the models using GridSearchCV to find the best hyperparameters. We'll compare baseline vs tuned performance.

In [ ]:
# Hyperparameter Tuning - Logistic Regression
print("\n" + "="*70)
print("HYPERPARAMETER TUNING - LOGISTIC REGRESSION")
print("="*70)

# Define parameter grid
lr_param_grid = {
    'C': [0.001, 0.01, 0.1, 1.0, 10.0, 100.0],
    'penalty': ['l2'],
    'solver': ['lbfgs', 'liblinear'],
    'max_iter': [1000]
}

# Base model
lr_base = LogisticRegression(class_weight='balanced', random_state=42)

# GridSearchCV with time series cross-validation
lr_search = GridSearchCV(
    lr_base,
    lr_param_grid,
    cv=tscv_sel,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1
)

# Prepare scaled data for LR
scaler_lr = StandardScaler()
X_scaled_lr = scaler_lr.fit_transform(X_selected_sorted)

# Fit grid search
print("Running grid search...")
lr_search.fit(X_scaled_lr, y_engineered_sorted)

# Best parameters
print(f"\nBest parameters: {lr_search.best_params_}")
print(f"Best cross-validation ROC-AUC: {lr_search.best_score_:.4f}")

# Store for later comparison
lr_params = lr_search.best_params_
tuning_results = {'Logistic Regression': {'baseline': roc_auc_lr_sel, 'tuned': lr_search.best_score_, 'params': lr_params}}

In [ ]:
# Hyperparameter Tuning - Random Forest
print("\n" + "="*70)
print("HYPERPARAMETER TUNING - RANDOM FOREST")
print("="*70)

# Define parameter grid
rf_param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 7, 10, None],
    'min_samples_split': [10, 15, 20],
    'min_samples_leaf': [4, 8, 12],
    'max_features': ['sqrt', 'log2']
}

# Base model
rf_base = RandomForestClassifier(class_weight='balanced', random_state=42)

# GridSearchCV with time series cross-validation
rf_search = GridSearchCV(
    rf_base,
    rf_param_grid,
    cv=tscv_sel,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1
)

# Fit grid search (no scaling needed for RF)
print("Running grid search...")
rf_search.fit(X_selected_sorted, y_engineered_sorted)

# Best parameters
print(f"\nBest parameters: {rf_search.best_params_}")
print(f"Best cross-validation ROC-AUC: {rf_search.best_score_:.4f}")

# Store results
rf_params = rf_search.best_params_
tuning_results['Random Forest'] = {'baseline': roc_auc_rf_sel, 'tuned': rf_search.best_score_, 'params': rf_params}

In [ ]:
# Hyperparameter Tuning - Gradient Boosting
print("\n" + "="*70)
print("HYPERPARAMETER TUNING - GRADIENT BOOSTING")
print("="*70)

# Define parameter grid
gb_param_grid = {
    'n_estimators': [50, 100, 200],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'max_depth': [3, 4, 5],
    'min_samples_split': [10, 15, 20],
    'min_samples_leaf': [4, 8, 12],
    'subsample': [0.7, 0.8, 0.9, 1.0]
}

# Base model
gb_base = GradientBoostingClassifier(random_state=42)

# GridSearchCV with time series cross-validation
gb_search = GridSearchCV(
    gb_base,
    gb_param_grid,
    cv=tscv_sel,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1
)

# Fit grid search (no scaling needed for GB)
print("Running grid search...")
gb_search.fit(X_selected_sorted, y_engineered_sorted)

# Best parameters
print(f"\nBest parameters: {gb_search.best_params_}")
print(f"Best cross-validation ROC-AUC: {gb_search.best_score_:.4f}")

# Store results
gb_params = gb_search.best_params_
tuning_results['Gradient Boosting'] = {'baseline': roc_auc_gb_sel, 'tuned': gb_search.best_score_, 'params': gb_params}

In [ ]:
# Hyperparameter Tuning - SVM
print("\n" + "="*70)
print("HYPERPARAMETER TUNING - SVM")
print("="*70)

# Define parameter grid
svm_param_grid = {
    'C': [0.1, 1.0, 10.0, 100.0],
    'gamma': ['scale', 'auto', 0.001, 0.01, 0.1],
    'kernel': ['rbf', 'poly'],
    'degree': [2, 3]  # Only used for poly kernel
}

# Base model
svm_base = SVC(class_weight='balanced', probability=True, random_state=42)

# GridSearchCV with time series cross-validation
svm_search = GridSearchCV(
    svm_base,
    svm_param_grid,
    cv=tscv_sel,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1
)

# Prepare scaled data for SVM
scaler_svm = StandardScaler()
X_scaled_svm = scaler_svm.fit_transform(X_selected_sorted)

# Fit grid search
print("Running grid search...")
svm_search.fit(X_scaled_svm, y_engineered_sorted)

# Best parameters
print(f"\nBest parameters: {svm_search.best_params_}")
print(f"Best cross-validation ROC-AUC: {svm_search.best_score_:.4f}")

# Store results
svm_params = svm_search.best_params_
tuning_results['SVM (RBF)'] = {'baseline': roc_auc_svm_sel, 'tuned': svm_search.best_score_, 'params': svm_params}

In [ ]:
# Comparison: Baseline vs Tuned Models
print("\n" + "="*70)
print("HYPERPARAMETER TUNING RESULTS COMPARISON")
print("="*70)

print("\nPerformance Comparison (ROC-AUC):")
print("-" * 70)
print(f"{'Model':<25} {'Baseline':>15} {'Tuned':>15} {'Improvement':>15}")
print("-" * 70)

for model_name, results in tuning_results.items():
    baseline_score = results['baseline']
    tuned_score = results['tuned']
    improvement = ((tuned_score - baseline_score) / baseline_score) * 100
    print(f"{model_name:<25} {baseline_score:>15.4f} {tuned_score:>15.4f} {improvement:>14.2f}%")

print("-" * 70)

print("\n\nBest Hyperparameters Found:")
print("-" * 70)
for model_name, results in tuning_results.items():
    print(f"\n{model_name}:")
    for param, value in results['params'].items():
        print(f"  {param}: {value}")

# Identify best overall model
best_model = max(tuning_results.items(), key=lambda x: x[1]['tuned'])
print("\n" + "="*70)
print(f"BEST MODEL: {best_model[0]} with ROC-AUC = {best_model[1]['tuned']:.4f}")
print("="*70)

In [ ]:
# Visualize Hyperparameter Tuning Impact
fig, ax = plt.subplots(figsize=(12, 6))

models = list(tuning_results.keys())
baseline_scores = [tuning_results[m]['baseline'] for m in models]
tuned_scores = [tuning_results[m]['tuned'] for m in models]

x = np.arange(len(models))
width = 0.35

bars1 = ax.bar(x - width/2, baseline_scores, width, label='Baseline', alpha=0.8, color='steelblue')
bars2 = ax.bar(x + width/2, tuned_scores, width, label='Tuned', alpha=0.8, color='darkorange')

ax.set_xlabel('Model', fontsize=12)
ax.set_ylabel('ROC-AUC Score', fontsize=12)
ax.set_title('Baseline vs Hyperparameter-Tuned Models', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(models, rotation=15, ha='right')
ax.legend()
ax.grid(axis='y', alpha=0.3)
ax.set_ylim([min(baseline_scores + tuned_scores) - 0.05, max(baseline_scores + tuned_scores) + 0.05])

# Add value labels on bars
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.4f}',
                ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

print("\nNote: Even small improvements in ROC-AUC can be significant with limited data.")